In [1]:
import equinox as eqx
import jax
import jax.numpy as jnp
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
)
from context_flux_no.models.multiphysics.hyperfluxfno import (
    HyperNeuralOperator,
)
from jaxtyping import Array, Float, PRNGKeyArray


jax.config.update("jax_default_device", jax.devices("gpu")[4])

E0710 17:45:09.422459  424803 platform_util.cc:270] Failed to create stream executor for device CUDA:0: : CUDA_ERROR_OUT_OF_MEMORY: out of memory


In [2]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=2,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FluxNO",
    target_network_kwargs=dict(
        stencil_widths=(10, 10), lift_dim=128, hidden_dim=128, depth=4
    ),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)

/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/hyperfluxfno/utils.py:53: UserWarning: TRecViTEncoder supports variable in_timesteps. The given 
                    in_timesteps value will be ignored.
  warnings.warn(
/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/nn/structured_linear.py:40: UserWarning: out_features is not divisible by num_blocks. Output vector 
            will be truncated to the requested size.
  warnings.warn("""out_features is not divisible by num_blocks. Output vector


2.77146


In [3]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=2,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FluxNO",
    target_network_kwargs=dict(
        stencil_widths=(10, 10), lift_dim=128, hidden_dim=128, depth=4
    ),
    lift_dim=64,
    width_hyper=128,
    blocks_hyper=16,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)

2.56397


In [5]:
@eqx.filter_jit
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


test_data = jax.random.normal(jax.random.key(0), (512, 21, 2, 100))

In [6]:
hyperfluxno(test_data[0, :20], (0.1, 0.01))

(20, 3, 100)


E0710 17:45:28.309974  425189 xtile_compiler.cc:399] Fusion: gemm_fusion_dot = f32[128,500]{1,0} fusion(a.1, bitcast.14), kind=kCustom, calls=gemm_fusion_dot_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0710 17:45:28.310053  425189 xtile_compiler.cc:401] Computation: gemm_fusion_dot_computation.clone {
  parameter_0 = f32[128,128]{1,0} parameter(0)
  parameter_1 = f32[128,500]{0,1} parameter(1)
  ROOT dot.1 = f32[128,500]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0710 17:45:28.315396  425204 xtile_compiler.cc:399] Fusion: gemm_fusion_do

(Array([[ 0.09306893,  0.01932453,  0.11297895,  0.05383476,  0.13953388,
          0.07984696,  0.05518745,  0.15022446,  0.06940588,  0.10365166,
          0.03493757,  0.07363091,  0.12053832,  0.01695365,  0.11219031,
          0.10357421,  0.03582637,  0.10120872,  0.04810455,  0.10572234,
          0.09302171,  0.04504388,  0.09028011,  0.1039194 ,  0.07672124,
          0.07232184,  0.09530669,  0.09043993,  0.09493047,  0.06449069,
          0.02154657,  0.06132896,  0.0651762 ,  0.0502397 ,  0.08219135,
          0.06556268,  0.1128578 ,  0.07719044,  0.14098197,  0.11579756,
          0.05387261,  0.09798107,  0.04676626,  0.07205829,  0.04606768,
          0.0231084 ,  0.10982864,  0.05983495,  0.03755468,  0.10604199,
          0.07307175,  0.07745682,  0.14997011,  0.02123786,  0.08839719,
          0.04442775,  0.08746317,  0.1545696 ,  0.06397621,  0.11582831,
          0.0764619 ,  0.10573048,  0.12939547,  0.06134258,  0.10980901,
          0.07176554, -0.00590755,  0.